# Week 2, Day 1: A three-way WrestleMania debate

**Lesson:** [week2/day1.ipynb](../../../week2/day1.ipynb), *Frontier model APIs, side by side*. **About 40 minutes.**

**What you practice here**

- Reaching several providers through one client class: an `OpenAI` client with each provider's base URL and key, and one for Ollama's local endpoint.
- Sending the same Chat Completions call to whichever client and model a speaker uses, and reading the reply's text.
- A conversation as a list of messages: in the lesson's two-bot chat each bot sees its own lines as `assistant` and the other's as `user`; for three speakers, one system prompt and one user prompt that holds the whole conversation.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week2/day1", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson sets two chatbots talking. Each has its own system prompt, and each sees its own past lines as the
assistant's and the other's as the user's. It ends with a challenge: bring in a third. With three speakers, the lesson
advises sending each one just one system prompt and one user prompt per turn, with the whole conversation so far in
the user prompt. Build both here, for three chatbots who debate the greatest WrestleMania main event. Their
personalities, `PERSONAS`, are given below.

First you make the clients, one per provider, as the lesson's cell of `OpenAI(api_key=..., base_url=...)` clients
does. `next_line` takes a client as its first parameter and `debate` a cast of clients, and they make the Chat
Completions calls themselves. The checks hand them stand-ins shaped like the real client: they record what you send
and answer with canned replies, so nothing is spent.

## What each call should give

```python
clients = make_clients({"OPENAI_API_KEY": "sk-proj-...", "GOOGLE_API_KEY": "AIza..."})
sorted(clients)               # ['gemini', 'ollama', 'openai']
clients["gemini"].base_url    # URL('https://generativelanguage.googleapis.com/v1beta/openai/')

conversation = [("Alex", "Hogan against Andre is the greatest main event ever."),
                ("Blake", "I can see why, Alex. The Silverdome was packed.")]
bot_messages("Blake", conversation)
# [{'role': 'system', 'content': PERSONAS['Blake']},
#  {'role': 'user', 'content': 'Hogan against Andre is the greatest main event ever.'},
#  {'role': 'assistant', 'content': 'I can see why, Alex. The Silverdome was packed.'}]

turn_messages("Charlie", conversation)
# [{'role': 'system', 'content': PERSONAS['Charlie']},
#  {'role': 'user', 'content': 'You are Charlie, in a debate with Alex and Blake.\n'
#                              'The conversation so far:\nAlex: Hogan against Andre ...\nBlake: I can see why ...\n'
#                              'Now say what you would like to say next, as Charlie.'}]

next_line(clients["gemini"], "Charlie", conversation, model="gemini-3.1-flash-lite")
# makes one request with that model and the messages from turn_messages("Charlie", conversation),
# and returns the reply's text, such as 'Actually, 93,173 fans saw it, a record at the time.'

debate({"Alex": (clients["openai"], "gpt-4.1-mini"), "Blake": (clients["ollama"], "llama3.2"),
        "Charlie": (clients["gemini"], "gemini-3.1-flash-lite")}, conversation, rounds=2)
# the two opening lines, then six new (name, text) pairs: Alex, Blake, Charlie, Alex, Blake, Charlie
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
from openai import OpenAI

## Given

The three speakers, with personalities in the lesson's style: a dict from each speaker's name to that speaker's
system prompt. Alex argues with everything, Blake looks for common ground, and Charlie settles arguments with facts.
Run this cell: your parts use `PERSONAS`. In Try it for real you will give a speaker a new personality.

In [ ]:
PERSONAS = {
    "Alex": "You are Alex, a snarky heel commentator who disagrees with everything and challenges every claim. "
            "You are in a debate with Blake and Charlie about the greatest WrestleMania main event.",
    "Blake": "You are Blake, a polite interviewer who finds common ground and calms everyone down. "
             "You are in a debate with Alex and Charlie about the greatest WrestleMania main event.",
    "Charlie": "You are Charlie, a wrestling historian who settles arguments with dates and attendances. "
               "You are in a debate with Alex and Blake about the greatest WrestleMania main event.",
}

### 1 · `make_clients(env)`

Makes one `OpenAI` client for each provider whose key is set, plus one for Ollama, and returns them in a dict keyed by
provider. `env` is a mapping of environment variables, such as `os.environ` after `load_dotenv(override=True)`: read
the keys from `env` only. Make:

- `"openai"`: a client with the key `env["OPENAI_API_KEY"]`, at OpenAI's default endpoint;
- `"anthropic"`, `"gemini"` and `"groq"`: a client at that provider's base URL, with its key from `env`
  (`ANTHROPIC_API_KEY`, `GOOGLE_API_KEY` and `GROQ_API_KEY`). The base URLs, from the lesson, are
  `https://api.anthropic.com/v1/`, `https://generativelanguage.googleapis.com/v1beta/openai/` and
  `https://api.groq.com/openai/v1`;
- `"ollama"`, always: a client at `http://localhost:11434/v1` with the key `"ollama"`.

Leave out any provider whose key is missing or empty. The lesson makes every client and only prints "not set (and
this is optional)", but a client made with no key quietly falls back to your `OPENAI_API_KEY`: it would send your
OpenAI key to that provider, and fail only when called. Making a client sends nothing, so the checks use the real
`OpenAI` here and read its base URL and key.

<details><summary>Hint</summary>

```python
OpenAI(api_key=env["OPENAI_API_KEY"])
OpenAI(base_url="https://api.anthropic.com/v1/", api_key=env["ANTHROPIC_API_KEY"])
OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
```
`env.get(name)` is `None` for a missing key and `""` for an empty one, and both are false in an `if`.

</details>

In [ ]:
def make_clients(env):
    # Write make_clients here, as described above.
    raise NotImplementedError("make_clients is not written yet")

In [ ]:
check("make_clients")

### 2 · `bot_messages(speaker, conversation)`

Builds the messages one bot sees in the lesson's two-bot conversation: its own lines as the assistant's, and the
other's as the user's. `conversation` is a list of `(name, text)` pairs, oldest first, and `speaker` is one of the
names in `PERSONAS`. Return a new list: a system message holding `PERSONAS[speaker]`, then one message per line, in
order, whose content is the line's text alone. A line by `speaker` becomes an `"assistant"` message, and any other
line a `"user"` message. This is what the lesson's `call_gpt` and `call_claude` build, each from its own side.

<details><summary>Hint</summary>

Each message is a dict such as `{"role": "assistant", "content": text}`. Compare each line's `name` with `speaker` to
pick its role.

</details>

In [ ]:
def bot_messages(speaker, conversation):
    # Write bot_messages here, as described above.
    raise NotImplementedError("bot_messages is not written yet")

In [ ]:
check("bot_messages")

### 3 · `turn_messages(speaker, conversation)`

Builds one turn's messages for three speakers: the speaker's system prompt, then one user prompt that holds the whole
conversation. Return a new list of two messages: a system message holding `PERSONAS[speaker]`, then a user message
that names `speaker`, holds every line of the conversation in order, each written `Name: text` on a line of its own,
and asks `speaker` what they would like to say next. The wording around the lines is yours. This is the lesson's
advice for its three-way challenge, as user and assistant roles alone cannot tell a speaker who said what.

<details><summary>Hint</summary>

The lesson's suggested user prompt, where `conversation` holds the lines joined with new lines:

```python
f"""You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Alex."""
```

</details>

In [ ]:
def turn_messages(speaker, conversation):
    # Write turn_messages here, as described above.
    raise NotImplementedError("turn_messages is not written yet")

In [ ]:
check("turn_messages")

### 4 · `next_line(client, speaker, conversation, model="gpt-4.1-mini")`

Asks one speaker for their next line in one call, and returns its text. Make one Chat Completions call with the model
`model` and the messages from `turn_messages(speaker, conversation)`, passing everything by name, as the SDK requires.
Return the reply's text, with the whitespace around it removed.

<details><summary>Hint</summary>

```python
response = client.chat.completions.create(model=model, messages=turn_messages(speaker, conversation))
return response.choices[0].message.content.strip()
```

</details>

In [ ]:
def next_line(client, speaker, conversation, model="gpt-4.1-mini"):
    # Write next_line here, as described above.
    raise NotImplementedError("next_line is not written yet")

In [ ]:
check("next_line")

### 5 · `debate(cast, conversation, rounds=3)`

Runs the three-way debate: in each round every speaker says one line, and each speaker hears every line said before
it. `cast` is a dict from a speaker's name (one of the names in `PERSONAS`) to a `(client, model)` pair, in speaking
order, so each speaker can be on a provider of its own. `conversation` is the opening list of `(name, text)` pairs. In
each of the `rounds` rounds, ask each speaker, in `cast`'s order, for its next line with `next_line` and its own client
and model, and add `(speaker, line)` to the conversation before you ask the next speaker. Return a new list: the
opening lines, then `rounds * len(cast)` new lines. Leave the list you were given unchanged.

<details><summary>Hint</summary>

Start from a copy, `conversation = list(conversation)`, then loop with `for _ in range(rounds):` and, inside it,
`for speaker, (client, model) in cast.items():`.

</details>

In [ ]:
def debate(cast, conversation, rounds=3):
    # Write debate here, as described above.
    raise NotImplementedError("debate is not written yet")

In [ ]:
check("debate")

## Check everything

The cell below runs all 9 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 2, Day 1 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

A round on `gpt-4.1-mini` and `claude-haiku-4-5` costs a fraction of a cent. Gemini and Groq have free tiers, and
Ollama is free but must be running (`ollama serve`, with `llama3.2` pulled).

1. Load your keys with `load_dotenv(override=True)`, build `clients = make_clients(os.environ)`, and print which
   providers you have.
2. The lesson's two-bot conversation: start a list `talk` with a line from Alex and a reply from Blake, then let Alex
   and Blake take turns for two rounds, each asking `gpt-4.1-mini` with `bot_messages(speaker, talk)` and adding its
   line to `talk`. Then print the roles in `bot_messages("Charlie", talk)`: Alex's and Blake's lines both reach Charlie
   as plain user messages, with nothing to say who said what. That is why the lesson switches to one user prompt.
3. Build a cast: Alex on `gpt-4.1-mini`, and Blake and Charlie on a second provider (`claude-haiku-4-5` or
   `gemini-3.1-flash-lite`) or on `llama3.2` through Ollama. Define an opening, such as
   `opening = [("Alex", "Hogan against Andre at WrestleMania III is the greatest main event of all time.")]`, run
   `debate(cast, opening, rounds=2)`, and print each line as `Name: text`.
4. Give one speaker a new personality by replacing its entry in `PERSONAS` (the lesson suggests a pessimist against
   an optimist), and run the debate again.
5. Optional: ask `gpt-5-nano` the lesson's coin puzzle with `reasoning_effort="minimal"`, then `"low"`, and compare
   the answers; and send one turn through LiteLLM's `completion`, printing its token count and its cost.

<details><summary>Stuck? One way to write it</summary>

```python
import os
from dotenv import load_dotenv

load_dotenv(override=True)
clients = make_clients(os.environ)
print("Providers:", ", ".join(clients))
```

```python
openai = clients["openai"]
talk = [("Alex", "Hogan against Andre at WrestleMania III is the greatest main event ever."),
        ("Blake", "I can see why you would say that, Alex!")]
for speaker in ["Alex", "Blake"] * 2:
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=bot_messages(speaker, talk))
    talk.append((speaker, response.choices[0].message.content))
    print(f"{speaker}: {talk[-1][1]}\n")

# Charlie joins: whose lines are these? Every other speaker's line is just "user"
for message in bot_messages("Charlie", talk):
    print(message["role"], "|", message["content"][:60])
```

```python
fallback = (clients["ollama"], "llama3.2")
cast = {"Alex": (clients["openai"], "gpt-4.1-mini"),
        "Blake": (clients["anthropic"], "claude-haiku-4-5") if "anthropic" in clients else fallback,
        "Charlie": (clients["gemini"], "gemini-3.1-flash-lite") if "gemini" in clients else fallback}
opening = [("Alex", "Hogan against Andre at WrestleMania III is the greatest main event of all time.")]
for name, text in debate(cast, opening, rounds=2):
    print(f"{name}: {text}\n")
```

```python
PERSONAS["Blake"] = ("You are Blake, a gloomy pessimist who finds every WrestleMania a letdown. "
                     "You are in a debate with Alex and Charlie about the greatest WrestleMania main event.")
for name, text in debate(cast, opening, rounds=1):
    print(f"{name}: {text}\n")
```

```python
# Optional: the lesson's puzzle, with less and then more reasoning
easy_puzzle = [{"role": "user", "content": "You toss 2 coins. One of them is heads. "
                                           "What's the probability the other is tails? Answer with the probability only."}]
for effort in ["minimal", "low"]:
    response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort=effort)
    print(effort, "->", response.choices[0].message.content)

# Optional: one turn through LiteLLM, with its token count and cost
from litellm import completion

response = completion(model="openai/gpt-4.1-mini", messages=turn_messages("Blake", opening))
print(response.choices[0].message.content)
print(response.usage.total_tokens, "tokens,", f"{response._hidden_params['response_cost'] * 100:.4f} cents")
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.